# Interactive Visualization of 3D Gaussian Splats with nerfstudio's gsplat renderer

Here we show how to use Kaolin's **camera conversion**, **ply/USD readers** and **interactive ipython visualizer** to view a 3D Guassian Splat model rendered with [nerfstudio's gsplat library](https://github.com/nerfstudio-project/gsplat). 

Specifically, we'll show how to interactively control aligned mesh and splat renderings using same camera conventions.

In [ ]:
!pip install -q matplotlib
!pip install gsplat

import copy
import gsplat.rendering
import ipywidgets
import ipyevents
import logging
import math
import os
import torch

import kaolin
from kaolin.render.camera import kaolin_camera_to_gsplat_nerfstudio
from kaolin.utils.bundled_data import SCANNED_TOYS_PATH, SCANNED_TOYS_NAMES, download_scanned_toys_dataset
from kaolin.utils.log import log_tensor
from kaolin.visualize.ipython import quick_viz

%load_ext autoreload
%autoreload 2
%matplotlib inline

kaolin.utils.log.default_log_setup(logging.DEBUG)
    
device = 'cuda'

### Preliminaries

Let's download sample data and install gsplat. Note that our data contains an aligned **3D Gaussian Splat** model and a **low-poly mesh** approximation.

In [ ]:
download_scanned_toys_dataset()  # only re-downloads if needed

### Loading Data

Now let's load a gaussian splat model and mesh into torch tensors. 

We'll examine tensors and their stats: note that gaussian `positions` and mesh `vertices` have similar stats, - as these two are already aligned.

In [ ]:
TOY_NAME = SCANNED_TOYS_NAMES[0]
GSMODEL = os.path.join(SCANNED_TOYS_PATH, f'{TOY_NAME}.ply')  # USD also downloaded and supported for splats
MESH = os.path.join(SCANNED_TOYS_PATH, f'mesh.{TOY_NAME}.usd')

gsmodel = kaolin.io.import_gaussiancloud(GSMODEL).to(device)
print(gsmodel.to_string(print_stats=True))

mesh = kaolin.io.import_mesh(MESH).to(device)
print(mesh.to_string(print_stats=True))

### Filter by Opacity

Our `GaussianSplatModel` class makes some other operations easy, for example filtering by a mask (let's filter by opacity).

In [ ]:
mask = gsmodel.opacities > 0.1
print(f'Keeping {mask.sum()} gaussians out of {mask.shape[0]}')

gsmodel = gsmodel[mask]
print(gsmodel)

### Basic Rendering (Aligned Mesh and Splats)

Now we'll use Kaolin's `easy_render` module to render mesh using Kaolin camera, and we will use kaolin's converter to render using nerfstudio gsplat library in the same coordinate frame.

In [ ]:
def render_with_gsplat(kal_cam, gsmodel):
    gsplat_cam_params = kaolin_camera_to_gsplat_nerfstudio(kal_cam)
    render_colors, render_alphas, info = gsplat.rendering.rasterization(
            gsmodel.positions,  # [N, 3]
            gsmodel.orientations,  # [N, 4]
            gsmodel.scales,  # [N, 3]
            gsmodel.opacities,  # [N]
            gsmodel.sh_coeff,  # [N, S, 3]
            sh_degree=gsmodel.sh_degree,  
            **gsplat_cam_params)
    return render_colors, render_alphas, info

kaolin_cam = kaolin.render.camera.Camera.from_args(
     eye=mesh.vertices.mean(dim=0) + torch.tensor([0, -1.3, 0], device=device), at=mesh.vertices.mean(dim=0), up=torch.tensor([0., 0.0, 1.0], device=device),
     fov=math.pi * 50 / 180, height=512, width=512).to(device)

colors, alphas, info = render_with_gsplat(kaolin_cam, gsmodel)
log_tensor(colors, 'colors', print_stats=True)
log_tensor(alphas, 'alphas', print_stats=True)

mesh_render = kaolin.render.easy_render.render_mesh(kaolin_cam, mesh)
log_tensor(mesh_render, 'mesh_render', print_stats=True)

kaolin.visualize.ipython.quick_viz(
    torch.cat([colors, mesh_render[kaolin.render.easy_render.RenderPass.albedo]], dim=0).permute(0, 3, 1, 2), inches=20).set_title('3D Gaussian Splat Rendering vs. Aligned Low-Poly Mesh')

### Interactive Visualization

Now we'll use Kaolin's ipython utility `IpyTurntableVisualizer` to allow interactive control of camera.

In [ ]:
def tst_render(in_cam):
    colors, _, _ = render_with_gsplat(in_cam, gsmodel)
    colors = (colors[0, ...] * 255).clip(0, 255).to(torch.uint8)
    return colors

def tst_render_lowres(in_cam):
    new_cam = copy.deepcopy(in_cam)
    new_cam.width = in_cam.width // 4
    new_cam.height = in_cam.height // 4
    return tst_render(new_cam)

visualizer = kaolin.visualize.IpyTurntableVisualizer(
    kaolin_cam.height, kaolin_cam.width, copy.deepcopy(kaolin_cam), tst_render, fast_render=tst_render_lowres,
    max_fps=24, world_up_axis=2, focus_at=torch.tensor([0, 0, 0.5], device=device))
visualizer.show()
